In [1]:
# Cell 1: Install core dependencies
%pip install python-dotenv langchain langchain-openai trafilatura -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2: Load environment variables and verify keys are present
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"
assert SERPER_API_KEY, "SERPER_API_KEY not found in .env"

print(f"OpenAI model : {OPENAI_MODEL}")
print(f"OpenAI key   : ...{OPENAI_API_KEY[-6:]}")
print(f"Serper key   : ...{SERPER_API_KEY[-6:]}")

OpenAI model : gpt-4o-mini
OpenAI key   : ...i-lLAA
Serper key   : ...d053d8


In [3]:
# Cell 2b: Disk-based cache for Serper and LLM calls
# Avoids re-calling paid APIs when re-running notebook cells.
# Cache lives in .cache/ as JSON files keyed by SHA-256 of the input.
import hashlib, json, pathlib

CACHE_DIR = pathlib.Path(".cache")
CACHE_DIR.mkdir(exist_ok=True)

def _cache_key(*parts) -> str:
    """Create a deterministic hash from arbitrary string parts."""
    raw = json.dumps(parts, sort_keys=True, ensure_ascii=True)
    return hashlib.sha256(raw.encode()).hexdigest()

def cache_get(namespace: str, *key_parts):
    """Return cached value or None if miss."""
    h = _cache_key(*key_parts)
    p = CACHE_DIR / namespace / f"{h}.json"
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return None

def cache_set(namespace: str, value, *key_parts):
    """Write value to cache."""
    h = _cache_key(*key_parts)
    d = CACHE_DIR / namespace
    d.mkdir(exist_ok=True)
    p = d / f"{h}.json"
    p.write_text(json.dumps(value, ensure_ascii=False, indent=1), encoding="utf-8")

def cache_stats():
    """Print cache stats per namespace."""
    if not CACHE_DIR.exists():
        print("Cache: empty")
        return
    for ns in sorted(CACHE_DIR.iterdir()):
        if ns.is_dir():
            files = list(ns.glob("*.json"))
            print(f"  {ns.name}: {len(files)} entries")

print(f"Cache dir: {CACHE_DIR.resolve()}")
cache_stats()

Cache dir: C:\Users\Abhishek A\Defining_Category\.cache
  llm_scoring: 47 entries
  llm_synthesis: 3 entries
  serper: 94 entries


In [ ]:
# Cell 3: Category config + comprehensive trusted source registry (per doc1 §3 + doc2)
# Every domain from doc2 is included, organized by tier for search prioritization

TEST_CATEGORY = "Account-Based Marketing"

# doc1 §3: maturity tag — determines currency thresholds for scoring
# "emerging" = 6-12 months, "evolving" = 18-24 months, "stable" = 36+ months
CATEGORY_MATURITY = "evolving"  # ABM is established but still shifting (ABM→ABX drift)

# doc1 §3: gather under ALL aliases
CATEGORY_ALIASES = [
    "Account-Based Marketing",
    "ABM",
    "Account-Based Marketing Platforms",
    "ABM platforms",
    "Account-Based Everything",
    "ABX",
    "Account-Based Experience",
]

# ── Trusted sites from doc2, organized by tier ──────────────────────────

# TIER 1: Major industry analysts — highest value, search individually/small batches
# NOTE: Google site: operator is unreliable with deep sub-paths.
# Use root domains where safe. Forrester is clean per doc2 —
# "no major review-platform subdivision to filter out."
# Gartner needs care (reviews/digital-markets excluded via DROP_URL_PATTERNS).
TIER1_SITES = [
    # Gartner — root subdomains + glossary paths
    "blogs.gartner.com",
    "gartner.com/en/articles",
    "gartner.com/en/marketing/glossary",
    "gartner.com/en/information-technology/glossary",
    "gartner.com/en/sales/glossary",
    # Forrester — root domain is safe (doc2: "unusually clean")
    "forrester.com",
    "gartner.com"
    "go.forrester.com",
    # IDC — root domain + blog subdomain
    "idc.com",
    "blogs.idc.com",
]

# TIER 2: Independent analysts — often more open access
TIER2_SITES = [
    "constellationr.com",
    "infotech.com",
    "451research.com",
    "spglobal.com/marketintelligence",
    "omdia.tech.informa.com",
    "hfsresearch.com",
    "isg-one.com",
    "everestgrp.com",
    "nucleusresearch.com",
    "dresneradvisory.com",
    "abiresearch.com",
    "gigaom.com",
    "aragonresearch.com",
    "moorinsightsstrategy.com",
    "pund-it.com",
    "enderlegroup.com",
    "jgoldassociates.com",
]

# TIER 2b: Domain-specialist analysts
TIER2B_SITES = [
    "kuppingercole.com",
    "barc.com",
    "frost.com",
    "enterprisemanagement.com",   # EMA
    "esg-global.com",
    "tag-cyber.com",
    "securosis.com",
    "colemanparkes.com",
]

# TIER 3: Trade publications (byline-level filter applies downstream in scoring)
TRADE_PUB_SITES = ["None"]

# TIER 4: Practitioner/consultancy publications
CONSULTANCY_SITES = [
    "mckinsey.com",
    "bcg.com",
    "bain.com",
    "deloitte.com/insights",
    "accenture.com",
    "ey.com",
    "kpmg.com",
    "a16z.com",
]

# TIER 5: Academic / standards
ACADEMIC_SITES = [
    "hbr.org",
    "sloanreview.mit.edu",
    "nist.gov",
]

# Combine all for reference
ALL_TRUSTED_SITES = (TIER1_SITES + TIER2_SITES + TIER2B_SITES +  ACADEMIC_SITES)

# ── doc1 §4: Known analyst author-hub URLs for this category ────────────
# These are direct crawl targets — single-URL goldmines per doc1.
ANALYST_HUB_URLS = [
    # Forrester ABM analysts
    "https://www.forrester.com/blogs/author/john_arnold/",
    "https://www.forrester.com/blogs/author/jessie_johnson/",
    "https://www.forrester.com/blogs/author/terry_flaherty/",
    # Gartner articles (curated, includes TOPO acquisitions)
    "https://www.gartner.com/en/articles/the-account-based-everything-framework",
    # ISG / Ventana Research — Keith Dawson on ABM
    "https://research.isg-one.com/analyst-perspectives/topic/intelligent-marketing",
]

# ── Explicit DROP patterns (per doc2 "What's deliberately not on this list") ──
DROP_URL_PATTERNS = [
    "/software-reviews/",       # Info-Tech SoftwareReviews = review platform
    "/compare/",                # head-to-head comparison pages
    "/products/",               # product review pages
    "gpivendorresources",       # Gartner vendor portal
    "gartner.com/reviews",      # Gartner Peer Insights
    "gartner.com/en/digital-markets",  # Capterra/GetApp/Software Advice
    "g2.com", "trustradius.com", "capterra.com", "getapp.com",
    "sourceforge.net", "goodfirms.co", "crozdesk.com",
    # Event/sponsor pages (low content value)
    "/sponsors/",
    "/event/",
]

# ── Google-operator exclusions (applied IN Serper query itself) ──────
# Makes Google filter junk BEFORE returning results, saving API credits.
SERPER_EXCLUDE_SITES = [
    "store.frost.com",              # Frost paywall store
    "my.idc.com",                   # IDC paywalled docs
    "info.idc.com",                 # IDC gated lead-gen content
    "view.frost.com",               # Frost gated viewer
    "hub.frost.com",                # Frost hub pages
    "web-assets.bcg.com",           # BCG PDF/asset CDN
    "keithdawson.isg-one.com",      # personal blog subdomain
    "portal.gigaom.com",            # GigaOm paywalled portal
    "linkedin.com",                  # LinkedIn - not analyst research
    "youtube.com",                  # YouTube - not analyst research
    "wikipedia.org",                # Wikipedia - not expert analysis
    "optimizely.com",               # Vendor sites
    "salesforce.com",                # Vendor sites
    "adobe.com",                   # Vendor sites
    "oracle.com",                  # Vendor sites
    "demandbase.com",               # Vendor sites
    "cognism.com",                 # Vendor sites
    "zoomforth.com",                # Vendor sites
    "factors.ai",                  # Vendor sites
    "influ2.com",                 # Vendor sites
    "mutinyhq.com",                # Vendor sites
    "marketone.com",                # Vendor sites
    "strategicabm.com",            # Vendor sites
    "clay.com",                   # Vendor sites
    "hginsights.com",               # Vendor sites
    "xgrowth.com.au",              # Vendor/agency sites
    "datalane.com",                # Vendor sites
]
SERPER_EXCLUDE_INURL = [
    "/wp-content/uploads/",         # WordPress uploaded PDFs/images
    "/content/dam/",                # CMS asset dirs (Accenture, Deloitte)
    "/docs/default-source/",        # ISG document library
    "/downloads/",                  # IDC/misc download dirs
    "event-pdf-generator",          # Forrester event PDFs
]

# Build exclusion string once — appended to every Serper query
_exc_parts = ["-filetype:pdf"]
_exc_parts += [f"-site:{s}" for s in SERPER_EXCLUDE_SITES]
_exc_parts += [f'-inurl:"{p}"' for p in SERPER_EXCLUDE_INURL]
SERPER_EXCLUSIONS = " ".join(_exc_parts)

# ── Currency thresholds (months) based on maturity ──────────────────────
CURRENCY_THRESHOLDS = {"emerging": 12, "evolving": 24, "stable": 36}
MAX_SOURCE_AGE_MONTHS = CURRENCY_THRESHOLDS[CATEGORY_MATURITY]

print(f"Category      : {TEST_CATEGORY}")
print(f"Maturity      : {CATEGORY_MATURITY} (max source age: {MAX_SOURCE_AGE_MONTHS} months)")
print(f"Aliases       : {len(CATEGORY_ALIASES)}")
print(f"Tier 1 sites  : {len(TIER1_SITES)} (Gartner, Forrester, IDC)")
print(f"Tier 2 sites  : {len(TIER2_SITES)} (independent analysts)")
print(f"Tier 2b sites : {len(TIER2B_SITES)} (domain specialists)")
print(f"Trade pubs    : {len(TRADE_PUB_SITES)}")
print(f"Consultancies : {len(CONSULTANCY_SITES)}")
print(f"Academic      : {len(ACADEMIC_SITES)}")
print(f"Total sites   : {len(ALL_TRUSTED_SITES)}")
print(f"Analyst hubs  : {len(ANALYST_HUB_URLS)}")
print(f"Drop patterns : {len(DROP_URL_PATTERNS)}")
print(f"Serper excl.  : {len(SERPER_EXCLUDE_SITES)} sites, {len(SERPER_EXCLUDE_INURL)} inurl, +pdf filter")
print(f"\nExclusion string ({len(SERPER_EXCLUSIONS)} chars):")
print(f"  {SERPER_EXCLUSIONS}")

Category      : Account-Based Marketing
Maturity      : evolving (max source age: 24 months)
Aliases       : 7
Tier 1 sites  : 9 (Gartner, Forrester, IDC)
Tier 2 sites  : 17 (independent analysts)
Tier 2b sites : 8 (domain specialists)
Trade pubs    : 1
Consultancies : 8
Academic      : 3
Total sites   : 37
Analyst hubs  : 5
Drop patterns : 15
Serper excl.  : 27 sites, 5 inurl, +pdf filter

Exclusion string (688 chars):
  -filetype:pdf -site:store.frost.com -site:my.idc.com -site:info.idc.com -site:view.frost.com -site:hub.frost.com -site:web-assets.bcg.com -site:keithdawson.isg-one.com -site:portal.gigaom.com -site:linkedin.com -site:youtube.com -site:wikipedia.org -site:optimizely.com -site:salesforce.com -site:adobe.com -site:oracle.com -site:demandbase.com -site:cognism.com -site:zoomforth.com -site:factors.ai -site:influ2.com -site:mutinyhq.com -site:marketone.com -site:strategicabm.com -site:clay.com -site:hginsights.com -site:xgrowth.com.au -site:datalane.com -inurl:"/wp-content

In [ ]:
# Cell 4: Focused Serper search — STRICT site-restricted search only
# ONLY searches the explicitly listed sites, no other domains allowed
import requests, time
from urllib.parse import urlparse

SEARCH_DELAY = 0.3  # seconds between Serper calls

def serper_search(query: str, api_key: str, num: int = 10, **kwargs) -> list[dict]:
    """Call Serper.dev Google Search API. Cached to disk."""
    cached = cache_get("serper", query, num, kwargs)
    if cached is not None:
        return cached
    payload = {"q": query, "num": num}
    payload.update(kwargs)
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=15,
    )
    resp.raise_for_status()
    results = resp.json().get("organic", [])
    cache_set("serper", results, query, num, kwargs)
    time.sleep(SEARCH_DELAY)
    return results

def batch_site_queries(sites: list[str], batch_size: int = 5) -> list[str]:
    """Create batches of site: queries for strict site restriction."""
    batches = []
    for i in range(0, len(sites), batch_size):
        chunk = sites[i : i + batch_size]
        # Create strict site: OR clause
        clause = " OR ".join(f"site:{s}" for s in chunk)
        batches.append(f"({clause})")
    return batches

def url_is_blocked(url: str) -> bool:
    """Check if URL matches any drop patterns."""
    url_lower = url.lower()
    for pattern in DROP_URL_PATTERNS:
        if pattern in url_lower:
            return True
    return False

def url_is_from_allowed_sites(url: str, allowed_sites: list[str]) -> bool:
    """STRICT: Only allow URLs from the explicitly listed sites."""
    from urllib.parse import urlparse
    hostname = urlparse(url).netloc.lower()
    
    # Check against allowed sites (exact match or subdomain)
    for allowed in allowed_sites:
        allowed_lower = allowed.lower()
        if hostname == allowed_lower or hostname.endswith('.' + allowed_lower):
            return True
    
    return False

def run_strict_search_pass(name: str, sites: list[str], aliases: list[str],
                          batch_size: int, seen: set, results: list,
                          num_per_query: int = 6):
    """STRICT search pass - ONLY returns results from listed sites."""
    batches = batch_site_queries(sites, batch_size=batch_size)
    queries_run = 0
    hits_added = 0
    blocked = 0
    cache_hits = 0
    unexpected_domains = []
    
    for alias in aliases:
        for site_clause in batches:
            # STRICT query: only search the specified sites
            query = f'{site_clause} "{alias}" {SERPER_EXCLUSIONS}'
            queries_run += 1
            
            try:
                was_cached = cache_get("serper", query, num_per_query, {}) is not None
                if was_cached:
                    cache_hits += 1
                    
                hits = serper_search(query, SERPER_API_KEY, num=num_per_query)
                
                for h in hits:
                    url = h.get("link", "")
                    if not url or url in seen:
                        continue
                    
                    # STRICT FILTER: Only allow from specified sites
                    if not url_is_from_allowed_sites(url, sites):
                        hostname = urlparse(url).netloc.lower()
                        if hostname not in [d.split('.')[0] for d in unexpected_domains]:
                            unexpected_domains.append(hostname)
                        blocked += 1
                        continue
                    
                    # Additional blocklist filter
                    if url_is_blocked(url):
                        blocked += 1
                        continue
                    
                    seen.add(url)
                    results.append({
                        "url": url,
                        "title": h.get("title", ""),
                        "snippet": h.get("snippet", ""),
                        "query_alias": alias,
                        "search_pass": name,
                    })
                    hits_added += 1
                    
            except Exception as e:
                print(f"    ✗ query failed: {e}") 
    
    # Report unexpected domains if any
    if unexpected_domains:
        print(f"    ⚠️  Blocked {len(unexpected_domains)} unexpected domains: {', '.join(unexpected_domains)}")
    
    cached_msg = f", {cache_hits} from cache" if cache_hits else ""
    print(f"  {name}: {queries_run} queries → {hits_added} URLs (blocked {blocked}{cached_msg})")
    return queries_run

# ── Focused aliases ──
FOCUSED_ALIASES = [
    "Account-Based Marketing",
    "ABM platforms",
]

# Secondary aliases for Tier 1 only (higher precision)
SECONDARY_ALIASES = [
    "Account-Based Marketing Platforms",
    "Account-Based Everything",
]

# ── Key Tier 2 sites (martech/ABM relevant only) ──
KEY_TIER2_SITES = [
    "constellationr.com",
    "isg-one.com",          # includes former Ventana Research
    "gigaom.com",
    "nucleusresearch.com",
    "aragonresearch.com",
]


seen_urls = set()
all_results = []
total_queries = 0

print("=" * 60)
print(f"STRICT SEARCH: {TEST_CATEGORY}")
print(f"ONLY searching explicitly listed sites")
print(f"Primary aliases: {FOCUSED_ALIASES}")
print(f"Secondary aliases (Tier 1 only): {SECONDARY_ALIASES}")
print("=" * 60)

# Pass 0: Analyst hub URLs (direct URLs, no search needed)
print("\n── Pass 0: Analyst author-hub URLs ──")
hub_added = 0
for hub_url in ANALYST_HUB_URLS:
    if hub_url not in seen_urls and not url_is_blocked(hub_url):
        seen_urls.add(hub_url)
        all_results.append({
            "url": hub_url,
            "title": f"[Hub] {hub_url.split('/')[-2] if hub_url.endswith('/') else hub_url.split('/')[-1]}",
            "snippet": "",
            "query_alias": TEST_CATEGORY,
            "search_pass": "AnalystHub",
        })
        hub_added += 1
print(f"  AnalystHub: {hub_added} direct URLs added")

# Pass 1: Tier 1 major analysts — primary aliases
print("\n── Pass 1: Tier 1 (Gartner, Forrester, IDC) — primary aliases ──")
total_queries += run_strict_search_pass(
    "Tier1-primary", TIER1_SITES, FOCUSED_ALIASES,
    batch_size=3, seen=seen_urls, results=all_results, num_per_query=6
)

# Pass 1b: Tier 1 major analysts — secondary aliases
print("\n── Pass 1b: Tier 1 — secondary aliases ──")
total_queries += run_strict_search_pass(
    "Tier1-secondary", TIER1_SITES, SECONDARY_ALIASES,
    batch_size=3, seen=seen_urls, results=all_results, num_per_query=6
)

# Pass 2: Key Tier 2 analysts — primary aliases only
print("\n── Pass 2: Key Tier 2 (Constellation, ISG, GigaOm, Nucleus, Aragon) ──")
total_queries += run_strict_search_pass(
    "Tier2-key", KEY_TIER2_SITES, FOCUSED_ALIASES,
    batch_size=5, seen=seen_urls, results=all_results, num_per_query=6
)

# Pass 2b: Key Tier 2 analysts — primary aliases only
print("\n── Pass 2: Key Tier 2 (Constellation, ISG, GigaOm, Nucleus, Aragon) ──")
total_queries += run_strict_search_pass(
    "Tier2-key", KEY_TIER2_SITES, SECONDARY_ALIASES,
    batch_size=5, seen=seen_urls, results=all_results, num_per_query=6
)

# Summary
print(f"\n{'=' * 60}")
print(f"TOTAL: {total_queries} Serper queries → {len(all_results)} unique URLs")
print(f"{'=' * 60}")

from collections import Counter
pass_counts = Counter(r["search_pass"] for r in all_results)
for pass_name, count in pass_counts.items():
    print(f"  {pass_name}: {count} URLs")

print(f"\nAll {len(all_results)} results:")
for i, r in enumerate(all_results):
    print(f"  {i+1}. [{r['search_pass']}] {r['title'][:70]}")
    print(f"     {r['url']}")

# Verify all results are from allowed sites
print(f"\n{'=' * 60}")
print("VERIFICATION: All URLs should be from explicitly listed sites")
print(f"{'=' * 60}")

from urllib.parse import urlparse
all_allowed_sites = TIER1_SITES + KEY_TIER2_SITES
verification_passed = True

for r in all_results:
    if not url_is_from_allowed_sites(r["url"], all_allowed_sites) and r["search_pass"] != "AnalystHub":
        hostname = urlparse(r["url"]).netloc.lower()
        print(f"❌ UNEXPECTED DOMAIN: {hostname} - {r['url']}")
        verification_passed = False

if verification_passed:
    print("✅ VERIFICATION PASSED: All URLs are from explicitly listed sites")
else:
    print("❌ VERIFICATION FAILED: Found URLs from unexpected domains")

cache_stats()

STRICT SEARCH: Account-Based Marketing
ONLY searching explicitly listed sites
Primary aliases: ['Account-Based Marketing', 'ABM platforms']
Secondary aliases (Tier 1 only): ['Account-Based Marketing Platforms', 'Account-Based Everything']

── Pass 0: Analyst author-hub URLs ──
  AnalystHub: 5 direct URLs added

── Pass 1: Tier 1 (Gartner, Forrester, IDC) — primary aliases ──
    ⚠️  Blocked 1 unexpected domains: gpivendorresources.gartner.com
  Tier1-primary: 6 queries → 21 URLs (blocked 1, 6 from cache)

── Pass 1b: Tier 1 — secondary aliases ──
    ⚠️  Blocked 4 unexpected domains: gpivendorresources.gartner.com, www.gartner.com, www.gartner.com, www.gartner.com
  Tier1-secondary: 6 queries → 14 URLs (blocked 4, 6 from cache)

── Pass 2: Key Tier 2 (Constellation, ISG, GigaOm, Nucleus, Aragon) ──
  Tier2-key: 2 queries → 9 URLs (blocked 0, 2 from cache)

TOTAL: 14 Serper queries → 49 unique URLs
  AnalystHub: 5 URLs
  Tier1-primary: 21 URLs
  Tier1-secondary: 14 URLs
  Tier2-key: 9 U

In [ ]:
# Cell 5: Pre-scraping filters + Trafilatura scraping
# Apply URL and date filters BEFORE scraping to save time and resources
import trafilatura
import time as _time
from datetime import datetime, timedelta
from urllib.parse import urlparse

SCRAPE_DELAY = 0.5  # seconds between fetches to be polite

def extract_article(url: str) -> dict | None:
    """Download and extract article content from a URL."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"_error": "fetch returned empty (403/timeout/JS-only)"}
        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            output_format="txt",
        )
        if not text:
            return {"_error": "extraction returned no text (template page?)"}
        meta = trafilatura.metadata.extract_metadata(downloaded)
        return {
            "url": url,
            "title": meta.title if meta else "",
            "author": meta.author if meta else "",
            "date": meta.date if meta else "",
            "text": text,
            "hostname": meta.sitename if meta else "",
        }
    except Exception as e:
        return {"_error": f"exception: {type(e).__name__}: {e}"}

def source_age_months(date_str: str) -> int | None:
    """Calculate source age in months from date string."""
    if not date_str:
        return None
    try:
        dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
        delta = datetime.now() - dt
        return int(delta.days / 30.44)
    except (ValueError, TypeError):
        return None

def should_scrape_url(url: str, title: str, search_pass: str) -> tuple[bool, str]:
    """Pre-scraping filter: decide if URL should be scraped based on criteria."""
    
    # 1. URL pattern filtering (same as before)
    url_lower = url.lower()
    for pattern in DROP_URL_PATTERNS:
        if pattern in url_lower:
            return False, f"DROP_URL_PATTERN: {pattern}"
    
    # 2. Domain filtering (redundant but safe)
    hostname = urlparse(url).netloc.lower()
    for blocked in SERPER_EXCLUDE_SITES:
        if blocked in hostname:
            return False, f"BLOCKED_DOMAIN: {blocked}"
    
    # 3. Content-type filtering based on URL patterns
    # Skip obvious non-content pages
    non_content_patterns = [
        "/page/", "/category/", "/tag/", "/author/",
        "/search", "/login", "/register", "/contact",
        "/privacy", "/terms", "/about", "/careers",
        "/events", "/webinars", "/podcasts", "/videos",
        "/download", "/pdf", "/whitepaper", "/ebook",
        "/trial", "/demo", "/pricing", "/buy",
        "/cart", "/checkout", "/payment", "/subscribe"
    ]
    
    for pattern in non_content_patterns:
        if pattern in url_lower:
            return False, f"NON_CONTENT_PATTERN: {pattern}"
    
    # 4. Title quality filtering
    if not title or len(title.strip()) < 10:
        return False, "SHORT_OR_MISSING_TITLE"
    
    # Skip obvious vendor marketing pages
    vendor_title_patterns = [
        "best", "top", "vs", "comparison", "review", "rating",
        "pricing", "cost", "trial", "demo", "free",
        "buy now", "get started", "sign up", "download"
    ]
    
    title_lower = title.lower()
    for pattern in vendor_title_patterns:
        if pattern in title_lower and len(title_lower.split()) < 8:
            return False, f"VENDOR_TITLE_PATTERN: {pattern}"
    
    # 5. Tier-based quality expectations
    # Higher tiers get more lenient treatment
    if search_pass.startswith("Tier1"):
        # Tier1: Allow most URLs, minimal filtering
        pass
    elif search_pass.startswith("Tier2"):
        # Tier2: Slightly more restrictive
        if "blog" not in url_lower and "article" not in url_lower and "research" not in url_lower:
            # Not obviously a content page
            return False, "TIER2_NON_CONTENT_URL"
    else:
        # Other tiers: More restrictive
        if not any(indicator in url_lower for indicator in ["blog", "article", "research", "insight", "analysis", "report"]):
            return False, "LOW_TIER_NON_CONTENT_URL"
    
    return True, "PASS"

def should_keep_scraped_content(article: dict, max_age_months: int) -> tuple[bool, str]:
    """Post-scraping filter: check if scraped content meets quality criteria."""
    
    # 1. Basic content validation
    if not article or "_error" in article:
        return False, f"SCRAPE_ERROR: {article.get('_error', 'unknown')}"
    
    text = article.get("text", "")
    if not text or len(text) < 300:
        return False, f"TOO_SHORT: {len(text)} chars"
    
    # 2. Currency/date filtering
    date_str = article.get("date")
    if date_str:
        age = source_age_months(date_str)
        if age is not None and age > max_age_months:
            return False, f"TOO_OLD: {age} months (max: {max_age_months})"
    
    # 3. Content quality filtering
    # Skip obvious marketing fluff
    marketing_phrases = [
        "best in class", "industry leading", "cutting edge", "state of the art",
        "revolutionary", "game changing", "breakthrough", "innovative",
        "competitive pricing", "affordable", "cost-effective",
        "free trial", "no credit card", "money back guarantee",
        "contact us today", "get started now", "sign up free"
    ]
    
    text_lower = text.lower()
    marketing_count = sum(1 for phrase in marketing_phrases if phrase in text_lower)
    
    # If too many marketing phrases relative to content length
    if marketing_count > 3 and len(text) < 1000:
        return False, f"TOO_MUCH_MARKETING: {marketing_count} phrases"
    
    # 4. Author byline quality (importance varies by tier)
    author = article.get("author", "")
    hostname = article.get("hostname", "")
    
    # For analyst sites, expect named authors
    if any(analyst in hostname for analyst in ["gartner", "forrester", "idc", "constellation", "isg-one"]):
        if not author or len(author.strip()) < 3:
            return False, "ANALYST_SITE_NO_AUTHOR"
    
    return True, "PASS"

# ── Apply pre-scraping filters ────────────────────────────────────────
print("Applying PRE-SCRAPING filters...")
print("=" * 60)

filtered_for_scraping = []
pre_scrape_filtered = []

for i, result in enumerate(all_results):
    url = result["url"]
    title = result["title"]
    search_pass = result["search_pass"]
    
    should_scrape, reason = should_scrape_url(url, title, search_pass)
    
    if should_scrape:
        filtered_for_scraping.append(result)
    else:
        pre_scrape_filtered.append({
            "url": url,
            "title": title,
            "reason": reason,
            "search_pass": search_pass
        })

print(f"Original URLs: {len(all_results)}")
print(f"Pre-scrape filtered: {len(pre_scrape_filtered)}")
print(f"Remaining for scraping: {len(filtered_for_scraping)}")

if pre_scrape_filtered:
    print(f"\nPre-scrape filtered URLs (first 10):")
    for f in pre_scrape_filtered[:10]:
        print(f"  ✗ {f['reason']}: {f['title'][:60]}")
        print(f"    {f['url']}")

# ── Scrape filtered URLs with Trafilatura ───────────────────────────────
print(f"\nScraping {len(filtered_for_scraping)} filtered URLs...")
print("=" * 60)

scraped_sources = []
scrape_failures = []

for i, result in enumerate(filtered_for_scraping):
    print(f"[{i+1}/{len(filtered_for_scraping)}] {result['url'][:80]}…", end=" ")
    
    article = extract_article(result["url"])
    
    # Apply post-scraping filters
    should_keep, keep_reason = should_keep_scraped_content(article, MAX_SOURCE_AGE_MONTHS)
    
    if should_keep:
        article["query_alias"] = result["query_alias"]
        article["search_pass"] = result["search_pass"]
        scraped_sources.append(article)
        print(f"✓ ({len(article['text'])} chars)")
    else:
        scrape_failures.append({
            "url": result["url"],
            "reason": keep_reason
        })
        print(f"✗ ({keep_reason})")
    
    _time.sleep(SCRAPE_DELAY)

print(f"\n{'='*60}")
print(f"FINAL SCRAPE RESULTS:")
print(f"  Original URLs: {len(all_results)}")
print(f"  Pre-scrape filtered: {len(pre_scrape_filtered)}")
print(f"  Attempted to scrape: {len(filtered_for_scraping)}")
print(f"  Successfully scraped: {len(scraped_sources)}")
print(f"  Post-scrape filtered: {len(scrape_failures)}")
print(f"  Overall success rate: {len(scraped_sources)/len(all_results)*100:.1f}%")

# Breakdown by filter type
if pre_scrape_filtered:
    pre_scrape_reasons = {}
    for f in pre_scrape_filtered:
        reason_type = f['reason'].split(':')[0]
        pre_scrape_reasons[reason_type] = pre_scrape_reasons.get(reason_type, 0) + 1
    
    print(f"\nPre-scrape filter breakdown:")
    for reason, count in pre_scrape_reasons.items():
        print(f"  {reason}: {count}")

if scrape_failures:
    post_scrape_reasons = {}
    for f in scrape_failures:
        reason_type = f['reason'].split(':')[0]
        post_scrape_reasons[reason_type] = post_scrape_reasons.get(reason_type, 0) + 1
    
    print(f"\nPost-scrape filter breakdown:")
    for reason, count in post_scrape_reasons.items():
        print(f"  {reason}: {count}")

# Quality breakdown by tier
print(f"\nFinal scraped sources by tier:")
tier_scrapes = {}
for s in scraped_sources:
    tier = s.get("search_pass", "Unknown")
    tier_scrapes[tier] = tier_scrapes.get(tier, 0) + 1

for tier, count in tier_scrapes.items():
    pct = count / len(scraped_sources) * 100 if scraped_sources else 0
    print(f"  {tier}: {count} sources ({pct:.1f}%)")

Applying PRE-SCRAPING filters...
Original URLs: 49
Pre-scrape filtered: 9
Remaining for scraping: 40

Pre-scrape filtered URLs (first 10):
  ✗ NON_CONTENT_PATTERN: /author/: [Hub] john_arnold
    https://www.forrester.com/blogs/author/john_arnold/
  ✗ NON_CONTENT_PATTERN: /author/: [Hub] jessie_johnson
    https://www.forrester.com/blogs/author/jessie_johnson/
  ✗ NON_CONTENT_PATTERN: /author/: [Hub] terry_flaherty
    https://www.forrester.com/blogs/author/terry_flaherty/
  ✗ NON_CONTENT_PATTERN: /category/: Account-Based Marketing (ABM) - Forrester
    https://www.forrester.com/blogs/category/account-based-marketing-abm/
  ✗ NON_CONTENT_PATTERN: /page/: Central and Eastern Europe Archives - Page 9 of 25 - IDC
    https://www.idc.com/resource-center/blog/resource-region/3_223/page/9/
  ✗ NON_CONTENT_PATTERN: /page/: Page 18 – IDC
    https://www.idc.com/page/18/?orderby=relevance&country=72
  ✗ NON_CONTENT_PATTERN: /page/: Page 16 – IDC
    https://www.idc.com/page/16/?keywords&utm_so

In [7]:
# Cell 6: Preview scraped sources
for i, s in enumerate(scraped_sources):
    print(f"--- Source {i+1} ---")
    print(f"  Title  : {s['title']}")
    print(f"  Author : {s['author']}")
    print(f"  Date   : {s['date']}")
    print(f"  Host   : {s['hostname']}")
    print(f"  Alias  : {s['query_alias']}")
    print(f"  Length  : {len(s['text'])} chars")
    print(f"  Preview: {s['text']}…")
    print()

--- Source 1 ---
  Title  : Distributing Responsibilities Between An Account-Based Marketing Center Of Excellence And Field Marketing | Forrester
  Author : Nora Conklin
  Date   : 2025-03-04
  Host   : Forrester
  Alias  : Account-Based Marketing
  Length  : 670 chars
  Preview: As account-based marketing (ABM) programs mature and expand globally, organizations are inevitably faced with questions about the role that field marketers play in delivering these programs. Field marketers can add significant value to ABM programs thanks to their proximity to local accounts and reps, as well as their deep knowledge of local markets. Relying heavily on field marketing rather than a centralized office makes more sense in some ABM scenarios than in others. In this report, we outline important considerations that determine what ABM activities field marketers should conduct and describe four approaches to designing an ABM center of excellence (COE).…

--- Source 2 ---
  Title  : AVEVA: Scaling Acc

In [8]:
# Cell 6b: Deduplicate near-identical scraped content
# Many sites (e.g. Info-Tech SoftwareReviews comparisons) return identical boilerplate.
# Dedup by hashing the first 500 chars of extracted text.
import hashlib as _hl

def _content_hash(text: str) -> str:
    return _hl.md5(text[:500].strip().lower().encode()).hexdigest()

_seen_hashes = set()
deduped_sources = []
dupes_removed = 0
for s in scraped_sources:
    h = _content_hash(s["text"])
    if h in _seen_hashes:
        dupes_removed += 1
        continue
    _seen_hashes.add(h)
    deduped_sources.append(s)

print(f"Before dedup: {len(scraped_sources)} sources")
print(f"Duplicates removed: {dupes_removed}")
print(f"After dedup: {len(deduped_sources)} unique sources")

# Replace scraped_sources so downstream cells use deduped list
scraped_sources = deduped_sources

Before dedup: 16 sources
Duplicates removed: 0
After dedup: 16 unique sources


In [10]:
# Cell 7: Pre-filter — DROP_URL_PATTERNS + content-level quality gates
# Applies both URL-pattern blocking and minimum content thresholds.
from datetime import datetime, timedelta
from urllib.parse import urlparse

def source_age_months(date_str: str) -> int | None:
    """Estimate source age in months from a date string. Returns None if unparseable."""
    if not date_str:
        return None
    try:
        dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
        delta = datetime.now() - dt
        return int(delta.days / 30.44)
    except (ValueError, TypeError):
        return None

def should_keep(source: dict) -> tuple[bool, str]:
    """Return (keep, reason) for a scraped source."""
    url = source["url"].lower()
    # URL-pattern drop
    for pattern in DROP_URL_PATTERNS:
        if pattern in url:
            return False, f"DROP pattern: {pattern}"
    # Minimum content length
    if len(source["text"]) < 300:
        return False, f"too short ({len(source['text'])} chars)"
    # Currency check — warn but don't drop (downstream scoring handles it)
    age = source_age_months(source.get("date"))
    if age is not None and age > MAX_SOURCE_AGE_MONTHS * 1.5:
        return False, f"too old ({age} months, threshold {MAX_SOURCE_AGE_MONTHS})"
    return True, "ok"

filtered_sources = []
dropped_sources = []
for s in scraped_sources:
    keep, reason = should_keep(s)
    if keep:
        filtered_sources.append(s)
    else:
        dropped_sources.append({"title": s.get("title", "?"), "url": s["url"], "reason": reason})

print(f"Kept {len(filtered_sources)} sources, dropped {len(dropped_sources)}")
if dropped_sources:
    print("\nDropped:")
    for d in dropped_sources:
        print(f"  ✗ {d['reason']}: {d['title'][:60]}")
        print(f"    {d['url']}")

print(f"\nFiltered sources:")
for i, s in enumerate(filtered_sources):
    age = source_age_months(s.get("date"))
    age_str = f"{age}mo" if age is not None else "?"
    print(f"  {i+1}. [{s.get('search_pass', '?')}] {s['title']}")
    print(f"     Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} ({age_str}) | {len(s['text'])} chars")
    print(f"     {s['url']}")
    print()

Kept 16 sources, dropped 0

Filtered sources:
  1. [Tier1-primary] Distributing Responsibilities Between An Account-Based Marketing Center Of Excellence And Field Marketing | Forrester
     Author: Nora Conklin | Date: 2025-03-04 (14mo) | 670 chars
     https://www.forrester.com/report/distributing-responsibilities-between-an-account-based-marketing-center-of-excellence-and-field-marketing/RES172068

  2. [Tier1-primary] AVEVA: Scaling Account-Based Marketing To Drive Customer-Centric Growth | Forrester
     Author: Conrad Mills | Date: 2026-01-29 (3mo) | 687 chars
     https://www.forrester.com/report/aveva-scaling-account-based-marketing-to-drive-customer-centric-growth/RES191150

  3. [Tier1-primary] How ABM advertising accelerates B2B growth
     Author: Roger Beharry Lall; Martin Herold | Date: 2025-11-13 (5mo) | 7888 chars
     https://www.idc.com/resource-center/blog/how-abm-advertising-accelerates-b2b-growth/

  4. [Tier1-primary] IDC - Boost Your ROI with Data-Driven Account-B

In [11]:
# Cell 8: LLM-based source quality scoring (per doc1 §5)
# Scores each source on: slot-fill, function-verbs, author credibility, currency, vendor diversity
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import json, time as _time

LLM_DELAY = 0.2  # seconds between LLM calls

class SourceScore(BaseModel):
    slot_definition: bool = Field(description="Contains a category definition")
    slot_capabilities: bool = Field(description="Lists core capabilities of the software")
    slot_boundaries: bool = Field(description="Distinguishes from adjacent/related categories")
    slot_buyer_use: bool = Field(description="Describes buyer persona or use case")
    slot_vendors: bool = Field(description="Names representative vendors/products")
    slots_filled: int = Field(description="Count of slots filled (0-5)")
    uses_function_verbs: bool = Field(description="Uses expert verbs like orchestrate, unify, score, route, match (vs SEO adjectives like better, smarter, faster)")
    vendor_count: int = Field(description="Number of distinct vendors/products mentioned")
    vendor_names: list[str] = Field(description="List of distinct vendor/product names found")
    is_sme_content: bool = Field(description="Appears to be written by or for subject-matter experts, not SEO/marketing fluff")
    byline_quality: str = Field(description="'named_analyst' if byline is a recognized analyst at a known firm, 'named_author' if any named author, 'no_byline' if anonymous/staff")
    single_vendor_bias: bool = Field(description="True if source primarily promotes a single vendor rather than providing neutral analysis")
    relevance_score: int = Field(description="1-10 overall relevance to defining the software category")
    reasoning: str = Field(description="Brief explanation of the score")

SCORING_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are evaluating a web source for its usefulness in defining the software category "{category}".

Score it on these criteria (doc1 §5):

1. SLOT-FILL: Does it contain (a) a definition, (b) core capabilities, (c) boundaries vs adjacent categories, (d) buyer/use case, (e) representative vendors? Count how many of these 5 slots it fills.

2. FUNCTION-VERBS: Does it use expert verbs (orchestrate, unify, score, route, match, segment, personalize, align) rather than SEO adjectives (better, smarter, faster, top, best)? This is one of the cleanest SME vs fluff filters.

3. BYLINE QUALITY: Evaluate the author byline.
   - "named_analyst" = recognized analyst at a known firm (Gartner, Forrester, Constellation, ISG, etc.)
   - "named_author" = has a named author but not clearly a recognized analyst
   - "no_byline" = anonymous, staff-writer, or no author attribution
   For trade publications (CIO, ZDNet, Diginomica, MarTech, etc.), only "named_analyst" or recognized SME bylines make the source trustworthy. Staff-writer SEO pieces should score lower.

4. VENDOR DIVERSITY & BIAS: How many distinct vendors are named? List them. A source naming 5+ vendors from different corporate families shows range. A source naming only its host vendor or a tight 2-3 from the same family signals selection bias. Flag single_vendor_bias if the source primarily promotes one vendor.

5. SME CONTENT: Is this analyst/expert content or marketing fluff? Consider the depth of analysis, presence of frameworks, and whether it reads as editorial work vs promotional material.

6. RELEVANCE: How useful is this source for writing a definitive category page (1-10)?
   - 8-10: Fills 4-5 slots, uses function-verbs, SME-authored, multi-vendor
   - 5-7: Fills 2-3 slots or has partial quality signals
   - 1-4: Off-topic, thin, or promotional

Return valid JSON matching the schema."""),
    ("human", """Source URL: {url}
Title: {title}
Author: {author}
Date: {date}
Host: {hostname}
Source age: {source_age}

Content (first 3000 chars):
{text}"""),
])

# Print the full prompt template once for review
print("SCORING PROMPT TEMPLATE:")
print("=" * 60)
for msg in SCORING_PROMPT.messages:
    print(f"\n── {msg.prompt.template[:20]}… ({type(msg).__name__}) ──")
print(f"\nInput variables: {SCORING_PROMPT.input_variables}")
print("=" * 60)

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0, api_key=OPENAI_API_KEY)
structured_llm = llm.with_structured_output(SourceScore)
chain = SCORING_PROMPT | structured_llm

scored_sources = []
cache_hits_llm = 0
for i, s in enumerate(filtered_sources):
    print(f"[{i+1}/{len(filtered_sources)}] Scoring: {s['title'][:60]}…", end=" ")
    text_trunc = s["text"][:3000]
    age = source_age_months(s.get("date"))
    age_str = f"{age} months" if age is not None else "unknown"

    invoke_params = {
        "category": TEST_CATEGORY,
        "url": s["url"],
        "title": s["title"] or "Unknown",
        "author": s["author"] or "Unknown",
        "date": s["date"] or "Unknown",
        "hostname": s["hostname"] or "Unknown",
        "source_age": age_str,
        "text": text_trunc,
    }
    cache_key_parts = ("scoring_v2", TEST_CATEGORY, s["url"], s["title"] or "Unknown",
                       s["author"] or "Unknown", s["date"] or "Unknown",
                       s["hostname"] or "Unknown", text_trunc)
    cached = cache_get("llm_scoring", *cache_key_parts)
    if cached is not None:
        score_data = cached["score"] if "score" in cached else cached
        score = SourceScore(**score_data)
        cache_hits_llm += 1
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10, slots={score.slots_filled}/5 (cached)")
        continue
    try:
        score = chain.invoke(invoke_params)
        # Cache both score and rendered prompt
        rendered_msgs = SCORING_PROMPT.format_messages(**invoke_params)
        rendered_prompt = "\n".join(f"[{m.type}] {m.content}" for m in rendered_msgs)
        cache_set("llm_scoring", {
            "score": score.model_dump(),
            "prompt": rendered_prompt,
        }, *cache_key_parts)
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10, slots={score.slots_filled}/5")
        _time.sleep(LLM_DELAY)
    except Exception as e:
        print(f"✗ {e}")

# Sort by relevance score descending
scored_sources.sort(key=lambda x: x["score"].relevance_score, reverse=True)
print(f"\nScored {len(scored_sources)} sources. Top sources:")
if cache_hits_llm:
    print(f"  ({cache_hits_llm} scores loaded from cache, {len(scored_sources) - cache_hits_llm} new LLM calls)")
cache_stats()

c:\Users\Abhishek A\Defining_Category\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SCORING PROMPT TEMPLATE:

── You are evaluating a… (SystemMessagePromptTemplate) ──

── Source URL: {url}
Ti… (HumanMessagePromptTemplate) ──

Input variables: ['author', 'category', 'date', 'hostname', 'source_age', 'text', 'title', 'url']
[1/16] Scoring: Distributing Responsibilities Between An Account-Based Marke… ✓ relevance=8/10, slots=4/5 (cached)
[2/16] Scoring: AVEVA: Scaling Account-Based Marketing To Drive Customer-Cen… ✓ relevance=8/10, slots=5/5 (cached)
[3/16] Scoring: How ABM advertising accelerates B2B growth… ✓ relevance=9/10, slots=5/5 (cached)
[4/16] Scoring: IDC - Boost Your ROI with Data-Driven Account-Based Marketin… ✓ relevance=4/10, slots=2/5 (cached)
[5/16] Scoring: Choose a Customer Data Platform That Amplifies Your Customer… ✓ relevance=8/10, slots=4/5 (cached)
[6/16] Scoring: The AI Experience Era: The Next Decade of Tech Marketing… ✓ relevance=2/10, slots=0/5
[7/16] Scoring: Converging Platforms For Greater Efficiency: The Rise Of Rev… ✓ relevance=10/10, slo

In [12]:
# Cell 9: Display scored sources ranked by relevance
for i, item in enumerate(scored_sources):
    s = item["source"]
    sc = item["score"]
    print(f"{'='*60}")
    print(f"#{i+1}  Relevance: {sc.relevance_score}/10 | Slots: {sc.slots_filled}/5 | Vendors: {sc.vendor_count}")
    print(f"  Title : {s['title']}")
    print(f"  Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} | Host: {s['hostname'] or 'n/a'}")
    print(f"  URL   : {s['url']}")
    print(f"  Byline: {sc.byline_quality} | Expert verbs: {sc.uses_function_verbs} | SME: {sc.is_sme_content} | Bias: {sc.single_vendor_bias}")
    print(f"  Slots → Def:{sc.slot_definition} Cap:{sc.slot_capabilities} Bound:{sc.slot_boundaries} Buyer:{sc.slot_buyer_use} Vendors:{sc.slot_vendors}")
    if sc.vendor_names:
        print(f"  Vendors named: {', '.join(sc.vendor_names)}")
    print(f"  Reasoning: {sc.reasoning}")
    print()

# Select top sources for synthesis (relevance >= 5 and at least 2 slots filled)
# Also exclude single-vendor-bias sources unless they're the only option
top_sources = [
    item for item in scored_sources
    if item["score"].relevance_score >= 5
    and item["score"].slots_filled >= 2
    and not item["score"].single_vendor_bias
]
# If too few, relax the bias filter
if len(top_sources) < 3:
    top_sources = [
        item for item in scored_sources
        if item["score"].relevance_score >= 5
        and item["score"].slots_filled >= 2
    ]

print(f"\n{'='*60}")
print(f"Sources qualifying for synthesis: {len(top_sources)} (relevance≥5, slots≥2, no single-vendor bias)")
if len(top_sources) < 3:
    print("⚠️  WARNING: Fewer than 3 qualifying sources — synthesis may be thin.")

#1  Relevance: 10/10 | Slots: 5/5 | Vendors: 12
  Title : Converging Platforms For Greater Efficiency: The Rise Of Revenue Marketing Platforms
  Author: Kelvin Gee | Date: 2024-07-25 | Host: Forrester
  URL   : https://www.forrester.com/blogs/converging-platforms-for-greater-efficiency-the-rise-of-revenue-marketing-platforms/
  Byline: named_analyst | Expert verbs: True | SME: True | Bias: False
  Slots → Def:True Cap:True Bound:True Buyer:True Vendors:True
  Vendors named: Vendor 1, Vendor 2, Vendor 3, Vendor 4, Vendor 5, Vendor 6, Vendor 7, Vendor 8, Vendor 9, Vendor 10, Vendor 11, Vendor 12
  Reasoning: The source provides a comprehensive definition of revenue marketing platforms, lists core capabilities, distinguishes it from adjacent categories like MAPs, describes buyer use cases, and names multiple vendors. It uses expert verbs throughout and is authored by a recognized analyst at Forrester, indicating high-quality SME content. Overall, it is highly relevant for defining the sof

In [ ]:
# Cell 11: LLM Synthesis - Generate comprehensive category page
# All passed websites go to LLM synthesizer to generate detailed category page
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import json, time as _time

class CategoryPage(BaseModel):
    """Structured output for comprehensive category page synthesis."""
    category_name: str = Field(description="Primary category name")
    aliases: list[str] = Field(description="Known aliases for this category")
    definition: str = Field(description="2-4 sentence category definition synthesized from multiple sources")
    core_capabilities: list[str] = Field(description="Detailed core software capabilities using function-verbs")
    boundaries: str = Field(description="Detailed explanation of what this category is NOT and how it differs from adjacent categories")
    buyer_use_case: str = Field(description="Comprehensive description of buyer personas and use cases")
    representative_vendors: list[str] = Field(description="Named vendors from across multiple sources with notes on source diversity")
    category_drift: str = Field(description="Detailed analysis of how analyst firms disagree on scope, naming, or existence")
    market_overview: str = Field(description="Market size, growth trends, and adoption patterns")
    implementation_considerations: str = Field(description="Key considerations for organizations implementing this software")
    vendor_landscape: str = Field(description="Analysis of vendor ecosystem and market positioning")
    future_trends: str = Field(description="Emerging trends and future direction of the category")
    integration_points: str = Field(description="How this category integrates with other enterprise systems")
    success_metrics: str = Field(description="Key metrics and KPIs for measuring success")
    common_challenges: str = Field(description="Typical challenges organizations face with this category")
    source_count: int = Field(description="Number of sources used in synthesis")
    confidence: str = Field(description="high/medium/low based on source coverage and consensus")

COMPREHENSIVE_SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are writing a comprehensive category definition page for Cuspera. The page must be written in Cuspera's editorial voice — no direct quotation from sources (copyright). Multi-source consensus carries definition.

COMPREHENSIVE CONTENT REQUIREMENTS:
- Each section should be substantial (3-6 paragraphs) with detailed insights
- Use specific examples, vendor names, and concrete scenarios
- Include quotes from named analysts where relevant (paraphrased, not direct)
- Provide actionable insights for buyers and implementers

DETAILED SECTION REQUIREMENTS:

1. DEFINITION: 2-4 sentences describing what the SOFTWARE does, not methodology. Focus on the technical capabilities and business function.

2. CORE CAPABILITIES: 6-10 detailed capabilities using function-verbs (orchestrate, unify, score, route, segment, personalize, align, prioritize, automate, enrich, match). For each capability, provide 1-2 sentences explaining what it does in practice.

3. BOUNDARIES: Comprehensive analysis of adjacent categories with specific examples. Name exact competitors and explain the key differentiators. Include vendor examples for each adjacent category.

4. BUYER/USE CASE: Detailed breakdown of buyer personas (CMO, VP Demand Gen, Revenue Marketing, Sales Ops, etc.), organization types (B2B enterprise, mid-market SaaS, manufacturing, tech, etc.), and specific use cases with scenarios.

5. REPRESENTATIVE VENDORS: Comprehensive vendor list with categorization (leaders, challengers, specialists) and notes on which sources mentioned each vendor. Note any vendor that appears in only one source.

6. MARKET OVERVIEW: Market size, growth rates, adoption patterns, maturity level, and geographic distribution if available.

7. IMPLEMENTATION CONSIDERATIONS: Technical requirements, integration needs, change management, timeline expectations, and resource requirements.

8. VENDOR LANDSCAPE: Analysis of market structure, consolidation trends, entry barriers, and competitive dynamics.

9. FUTURE TRENDS: Emerging technologies, evolving buyer expectations, and category evolution over next 2-3 years.

10. INTEGRATION POINTS: How this category connects with CRM, ERP, marketing automation, sales enablement, and other enterprise systems.

11. SUCCESS METRICS: Specific KPIs, measurement approaches, and ROI expectations.

12. COMMON CHALLENGES: Implementation hurdles, adoption barriers, and mitigation strategies.

13. CATEGORY DRIFT: Detailed analysis of analyst disagreements with specific firm positions and naming conventions.

RULES FROM DOC1 §6:
- Fill five primary slots first: definition, core capabilities, boundaries, buyer/use case, representative vendors
- Use function-verbs, NOT benefit-adjectives (better, smarter, faster, top, best)
- Be concrete and specific with vendor names and examples
- Address category drift with specific firm names
- If analysts disagree on category existence, acknowledge directly

Category: {category}
Maturity: {maturity}
Aliases: {aliases}"""),
    ("human", """Here are the top-scoring sources to synthesize from:

{sources_text}

Synthesize these into a comprehensive category page with all detailed sections. Each section should provide substantial, actionable content. Return structured JSON."""),
])

# Prepare sources for synthesis
if not top_sources:
    print("❌ ERROR: No sources available for synthesis!")
    print("Check earlier cells for filtering issues.")
else:
    print(f"Synthesizing comprehensive category page from {len(top_sources)} high-quality sources...")
    print("=" * 70)
    
    # Prepare sources block with more detail
    sources_block = ""
    for i, item in enumerate(top_sources):
        s = item["source"]
        sc = item["score"]
        sources_block += f"\n--- SOURCE {i+1} (relevance {sc.relevance_score}/10, slots {sc.slots_filled}/5) ---\n"
        sources_block += f"Title: {s['title']}\n"
        sources_block += f"Author: {s['author'] or 'Unknown'} | Host: {s['hostname'] or 'Unknown'} | Date: {s['date'] or 'Unknown'}\n"
        sources_block += f"Byline quality: {sc.byline_quality} | Vendors named: {', '.join(sc.vendor_names) if sc.vendor_names else 'none'}\n"
        sources_block += f"Slots filled: Definition={sc.slot_definition} Capabilities={sc.slot_capabilities} Boundaries={sc.slot_boundaries} Buyer={sc.slot_buyer_use} Vendors={sc.slot_vendors}\n"
        sources_block += f"Content:\n{s['text'][:6000]}\n"  # More content for detailed synthesis
    
    # Check synthesis cache
    synth_cache_key = ("comprehensive_synthesis", TEST_CATEGORY, ", ".join(CATEGORY_ALIASES), sources_block)
    cached_synth = cache_get("llm_synthesis", *synth_cache_key)
    
    if cached_synth is not None:
        category_page = CategoryPage(**cached_synth)
        print("✓ Comprehensive synthesis loaded from cache!")
    else:
        # Generate comprehensive synthesis
        synth_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2, api_key=OPENAI_API_KEY)
        synth_chain = COMPREHENSIVE_SYNTHESIS_PROMPT | synth_llm.with_structured_output(CategoryPage)
        
        try:
            category_page = synth_chain.invoke({
                "category": TEST_CATEGORY,
                "maturity": CATEGORY_MATURITY,
                "aliases": ", ".join(CATEGORY_ALIASES),
                "sources_text": sources_block,
            })
            
            # Cache result
            cache_set("llm_synthesis", category_page.model_dump(), *synth_cache_key)
            print("✓ Comprehensive synthesis complete (cached for next run)!")
            
        except Exception as e:
            print(f"❌ Comprehensive synthesis failed: {e}")
            category_page = None
    
    # Display results if synthesis succeeded
    if category_page:
        print(f"\n{'=' * 70}")
        print(f"  COMPREHENSIVE CATEGORY PAGE: {category_page.category_name}")
        print(f"{'=' * 70}")
        print(f"\nAliases: {', '.join(category_page.aliases)}")
        print(f"Confidence: {category_page.confidence} | Sources used: {category_page.source_count}")
        
        print(f"\n{'─' * 70}")
        print("1. DEFINITION")
        print(f"{'─' * 70}")
        print(category_page.definition)
        
        print(f"\n{'─' * 70}")
        print("2. CORE CAPABILITIES")
        print(f"{'─' * 70}")
        for cap in category_page.core_capabilities:
            print(f"  • {cap}")
        
        print(f"\n{'─' * 70}")
        print("3. BOUNDARIES")
        print(f"{'─' * 70}")
        print(category_page.boundaries)
        
        print(f"\n{'─' * 70}")
        print("4. BUYER / USE CASE")
        print(f"{'─' * 70}")
        print(category_page.buyer_use_case)
        
        print(f"\n{'─' * 70}")
        print("5. REPRESENTATIVE VENDORS")
        print(f"{'─' * 70}")
        for v in category_page.representative_vendors:
            print(f"  • {v}")
        
        print(f"\n{'─' * 70}")
        print("6. MARKET OVERVIEW")
        print(f"{'─' * 70}")
        print(category_page.market_overview)
        
        print(f"\n{'─' * 70}")
        print("7. IMPLEMENTATION CONSIDERATIONS")
        print(f"{'─' * 70}")
        print(category_page.implementation_considerations)
        
        print(f"\n{'─' * 70}")
        print("8. VENDOR LANDSCAPE")
        print(f"{'─' * 70}")
        print(category_page.vendor_landscape)
        
        print(f"\n{'─' * 70}")
        print("9. FUTURE TRENDS")
        print(f"{'─' * 70}")
        print(category_page.future_trends)
        
        print(f"\n{'─' * 70}")
        print("10. INTEGRATION POINTS")
        print(f"{'─' * 70}")
        print(category_page.integration_points)
        
        print(f"\n{'─' * 70}")
        print("11. SUCCESS METRICS")
        print(f"{'─' * 70}")
        print(category_page.success_metrics)
        
        print(f"\n{'─' * 70}")
        print("12. COMMON CHALLENGES")
        print(f"{'─' * 70}")
        print(category_page.common_challenges)
        
        if category_page.category_drift:
            print(f"\n{'─' * 70}")
            print("13. CATEGORY DRIFT / ANALYST DISAGREEMENT")
            print(f"{'─' * 70}")
            print(category_page.category_drift)
        
        print(f"\n{'=' * 70}")
        print("Sources used in synthesis:")
        for i, item in enumerate(top_sources):
            s = item["source"]
            sc = item["score"]
            print(f"  [{i+1}] {s['title']} — {s['author'] or 'n/a'} ({s['hostname'] or 'n/a'})")
            print(f"      Relevance: {sc.relevance_score}/10, Byline: {sc.byline_quality}, Vendors: {len(sc.vendor_names)}")

cache_stats()

Synthesizing comprehensive category page from 5 high-quality sources...
✓ Comprehensive synthesis complete (cached for next run)!

  COMPREHENSIVE CATEGORY PAGE: Account-Based Marketing

Aliases: Account-Based Marketing, ABM, Account-Based Marketing Platforms, ABM platforms, Account-Based Everything, ABX, Account-Based Experience
Confidence: high | Sources used: 4

──────────────────────────────────────────────────────────────────────
1. DEFINITION
──────────────────────────────────────────────────────────────────────
Account-Based Marketing (ABM) is a strategic approach that targets specific accounts rather than individual leads, focusing on engaging entire buying groups through personalized marketing efforts. This software enables organizations to orchestrate multi-channel campaigns, unify data across platforms, and deliver tailored content to decision-makers, ultimately driving higher engagement and conversion rates within key accounts.

─────────────────────────────────────────────

In [14]:
# Cell 11: Display the final synthesized category page + export to JSON
import json, pathlib

print(f"{'='*70}")
print(f"  CATEGORY PAGE: {category_page.category_name}")
print(f"{'='*70}")
print(f"\nAliases: {', '.join(category_page.aliases)}")
print(f"Confidence: {category_page.confidence} | Sources used: {category_page.source_count}")

print(f"\n{'─'*70}")
print("1. DEFINITION")
print(f"{'─'*70}")
print(category_page.definition)

print(f"\n{'─'*70}")
print("2. CORE CAPABILITIES")
print(f"{'─'*70}")
for cap in category_page.core_capabilities:
    print(f"  • {cap}")

print(f"\n{'─'*70}")
print("3. BOUNDARIES")
print(f"{'─'*70}")
print(category_page.boundaries)

print(f"\n{'─'*70}")
print("4. BUYER / USE CASE")
print(f"{'─'*70}")
print(category_page.buyer_use_case)

print(f"\n{'─'*70}")
print("5. REPRESENTATIVE VENDORS")
print(f"{'─'*70}")
for v in category_page.representative_vendors:
    print(f"  • {v}")

if category_page.category_drift:
    print(f"\n{'─'*70}")
    print("6. CATEGORY DRIFT / ANALYST DISAGREEMENT")
    print(f"{'─'*70}")
    print(category_page.category_drift)

print(f"\n{'='*70}")
print("Sources used:")
for i, item in enumerate(top_sources):
    s = item["source"]
    sc = item["score"]
    print(f"  [{i+1}] {s['title']} — {s['author'] or 'n/a'} ({s['hostname'] or 'n/a'})")
    print(f"      {s['url']}  [relevance {sc.relevance_score}/10, byline: {sc.byline_quality}]")

# ── Export results to JSON for persistence ──────────────────────────────
output_dir = pathlib.Path("output")
output_dir.mkdir(exist_ok=True)

# Export category page
cat_slug = TEST_CATEGORY.lower().replace(" ", "_").replace("-", "_")
page_path = output_dir / f"{cat_slug}_page.json"
page_path.write_text(json.dumps(category_page.model_dump(), indent=2, ensure_ascii=False), encoding="utf-8")

# Export scored sources (for audit trail)
scores_path = output_dir / f"{cat_slug}_scores.json"
scores_export = []
for item in scored_sources:
    s = item["source"]
    sc = item["score"]
    scores_export.append({
        "url": s["url"],
        "title": s["title"],
        "author": s["author"],
        "date": s["date"],
        "hostname": s["hostname"],
        "search_pass": s.get("search_pass", ""),
        "text_length": len(s["text"]),
        "score": sc.model_dump(),
    })
scores_path.write_text(json.dumps(scores_export, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\n{'='*70}")
print(f"Exported:")
print(f"  Category page → {page_path}")
print(f"  Scored sources ({len(scores_export)}) → {scores_path}")

  CATEGORY PAGE: Account-Based Marketing

Aliases: Account-Based Marketing, ABM, Account-Based Marketing Platforms, ABM platforms, Account-Based Everything, ABX, Account-Based Experience
Confidence: high | Sources used: 4

──────────────────────────────────────────────────────────────────────
1. DEFINITION
──────────────────────────────────────────────────────────────────────
Account-Based Marketing (ABM) is a strategic approach that targets specific accounts rather than individual leads, focusing on engaging entire buying groups through personalized marketing efforts. This software enables organizations to orchestrate multi-channel campaigns, unify data across platforms, and deliver tailored content to decision-makers, ultimately driving higher engagement and conversion rates within key accounts.

──────────────────────────────────────────────────────────────────────
2. CORE CAPABILITIES
──────────────────────────────────────────────────────────────────────
  • Orchestrate multi-chann

In [15]:
# Cell 12: Second-pass review placeholder (per doc1 §1)
# "Two passes per category. First pass casts the net wide...
#  Second pass to check for derivative sources, and understand what couldn't be pinned down."
#
# TODO: Implement second pass:
# 1. Review the category_page output for gaps (e.g. missing vendors, weak boundaries)
# 2. Generate targeted follow-up queries for specific gaps
# 3. Check if any sources are derivative (citing the same underlying analyst report)
# 4. Flag what couldn't be pinned down for manual review
#
# For now, print a gap analysis based on the synthesis output.

gaps = []
if category_page.confidence != "high":
    gaps.append(f"Confidence is '{category_page.confidence}' — may need more sources")
if len(category_page.representative_vendors) < 5:
    gaps.append(f"Only {len(category_page.representative_vendors)} vendors — aim for 5+")
if not category_page.category_drift:
    gaps.append("No category drift detected — verify manually that analysts agree")
if len(category_page.core_capabilities) < 5:
    gaps.append(f"Only {len(category_page.core_capabilities)} capabilities listed — may be incomplete")
if len(top_sources) < 5:
    gaps.append(f"Only {len(top_sources)} qualifying sources — thin evidence base")

# Check source diversity
source_hosts = set()
for item in top_sources:
    host = item["source"].get("hostname", "")
    if host:
        source_hosts.add(host.lower().split("|")[0].strip())
if len(source_hosts) < 3:
    gaps.append(f"Sources from only {len(source_hosts)} distinct hosts — low diversity")

# Check for Tier 1 presence
tier1_found = any(
    any(t1 in item["source"]["url"] for t1 in ["forrester.com", "gartner.com", "idc.com"])
    for item in top_sources
)
if not tier1_found:
    gaps.append("⚠️  No Tier 1 analyst sources (Gartner/Forrester/IDC) in synthesis — major gap")

print(f"{'='*60}")
print("SECOND-PASS GAP ANALYSIS")
print(f"{'='*60}")
if gaps:
    for g in gaps:
        print(f"  ⚠️  {g}")
    print(f"\n{len(gaps)} gaps identified — consider targeted follow-up searches.")
else:
    print("  ✓ No major gaps detected.")
print(f"\nSource host diversity: {', '.join(sorted(source_hosts))}")

SECOND-PASS GAP ANALYSIS
  ✓ No major gaps detected.

Source host diversity: constellationr, forrester, idc
